# FinCast Prediction Quality Analysis

This notebook analyzes three different questions:

1. **Training and validation labels:** what is the class balance, per-ticker spread, and majority baseline?
2. **Validation behavior:** does the model beat the validation majority class, or does it mostly emit one label?
3. **Live prediction behavior:** how accurate are settled rows in `predictions_log.jsonl`, by ticker and by predicted/actual move?

A live prediction is scored only after `actual_move_bin` is available. Repeated predictions for the same ticker/date are reduced to the latest logged run.

In [ ]:
import json
import math
import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
MOVE_BINS = ['strong_down', 'down', 'flat', 'up', 'strong_up']

def majority_accuracy(labels):
    counts = pd.Series(labels).dropna().value_counts()
    if counts.empty:
        return np.nan, 'unavailable'
    return counts.iloc[0] / counts.sum(), counts.index[0]

def entropy_from_counts(counts):
    probabilities = np.asarray(counts, dtype=float)
    total = probabilities.sum()
    if total <= 0:
        return 0.0
    probabilities = probabilities[probabilities > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())

def locate_base_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'news_scraper', cwd / '..', cwd / '..' / '..']
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'data' / 'train_data' / 'combined').exists() and (candidate / 'predictions_log.jsonl').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate news_scraper from {cwd}')

BASE_DIR = locate_base_dir()
TRAIN_PATH = BASE_DIR / 'data' / 'train_data' / 'combined' / 'train.jsonl'
VAL_PATH = BASE_DIR / 'data' / 'train_data' / 'combined' / 'val.jsonl'
PREDICTIONS_PATH = BASE_DIR / 'predictions_log.jsonl'
print('BASE_DIR:', BASE_DIR)
print('TRAIN_PATH:', TRAIN_PATH, 'exists=', TRAIN_PATH.exists())
print('VAL_PATH:', VAL_PATH, 'exists=', VAL_PATH.exists())
print('PREDICTIONS_PATH:', PREDICTIONS_PATH, 'exists=', PREDICTIONS_PATH.exists())

## 1. Load and normalize the three data sources

The training and validation files use chat records. The assistant message is the ground-truth `move_bin`. The prediction log contains model output, actual outcome, percentage change, and news count.

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as fh:
        for line_number, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f'Skipping invalid JSON at {path}:{line_number}: {exc}')
    return rows

def extract_move_bin_from_record(record):
    messages = record.get('messages', [])
    if len(messages) < 3:
        return None
    try:
        payload = json.loads(messages[-1].get('content', '{}'))
    except (TypeError, json.JSONDecodeError):
        return None
    value = payload.get('move_bin')
    return value if value in MOVE_BINS else None

def extract_ticker_from_record(record):
    meta = record.get('_meta', {})
    if meta.get('ticker'):
        return str(meta['ticker']).upper()
    user_content = '\n'.join(str(message.get('content', '')) for message in record.get('messages', []) if message.get('role') == 'user')
    match = re.search(r'(?:^|\n)Ticker:\s*([A-Za-z0-9_.-]+)', user_content)
    return match.group(1).upper() if match else 'UNKNOWN'

def records_to_frame(records, split_name):
    rows = []
    for index, record in enumerate(records):
        rows.append({
            'row_id': index,
            'split': split_name,
            'ticker': extract_ticker_from_record(record),
            'date': record.get('_meta', {}).get('date'),
            'label': extract_move_bin_from_record(record),
        })
    return pd.DataFrame(rows)

train_df = records_to_frame(read_jsonl(TRAIN_PATH), 'train')
val_df = records_to_frame(read_jsonl(VAL_PATH), 'validation')
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

prediction_rows = read_jsonl(PREDICTIONS_PATH)
pred_log_df = pd.DataFrame(prediction_rows)
if not pred_log_df.empty:
    pred_log_df['logged_at'] = pd.to_datetime(pred_log_df.get('logged_at'), errors='coerce', utc=True)
    pred_log_df['date'] = pd.to_datetime(pred_log_df.get('date'), errors='coerce')
    for column in ['predicted_move_bin', 'actual_move_bin']:
        if column in pred_log_df:
            pred_log_df[column] = pred_log_df[column].astype('string')
    pred_log_df['ticker'] = pred_log_df['ticker'].astype('string').str.upper()
    pred_log_df['actual_change_1d'] = pd.to_numeric(pred_log_df.get('actual_change_1d'), errors='coerce')
    pred_log_df['num_news_items'] = pd.to_numeric(pred_log_df.get('num_news_items'), errors='coerce')
    pred_log_df['settled'] = pred_log_df['actual_move_bin'].isin(MOVE_BINS)
    pred_df = pred_log_df.sort_values('logged_at').drop_duplicates(['ticker', 'date'], keep='last').reset_index(drop=True)
else:
    pred_df = pred_log_df.copy()

print(f'Train rows: {len(train_df):,}; validation rows: {len(val_df):,}')
print(f'Prediction log rows after latest-run deduplication: {len(pred_df):,}')
if not pred_df.empty:
    print(f'Settled prediction rows: {pred_df["settled"].sum():,}; unsettled: {(~pred_df["settled"]).sum():,}')
display(train_val_df.head())
display(pred_df.head())

## 2. Repeat-run consistency

The log can contain multiple predictions for the same ticker and forecast date. For production accuracy, later rows are reduced to the latest run; this section keeps all raw runs to check whether repeated inference is stable or whether the label changes between runs.

Agreement is measured at the ticker/date-group level. A changed label is an audit signal, not automatically a model error: news inputs, article counts, market context, or model state may have changed between runs. The latest-run deduplicated frame is used by the later accuracy sections so repeated logging does not overweight one forecast.

In [ ]:
if pred_log_df.empty:
    print('No prediction-log rows are available for repeat-run analysis.')
else:
    repeat_groups = []
    for (ticker, date), group in pred_log_df.sort_values('logged_at').groupby(['ticker', 'date'], dropna=False):
        labels = group['predicted_move_bin'].dropna().astype(str)
        news_counts = group['num_news_items'].dropna()
        repeat_groups.append({
            'ticker': ticker,
            'date': date,
            'runs': len(group),
            'unique_labels': labels.nunique(),
            'consistent': labels.nunique() <= 1,
            'first_label': labels.iloc[0] if len(labels) else None,
            'latest_label': labels.iloc[-1] if len(labels) else None,
            'news_count_changed': news_counts.nunique() > 1,
        })

    repeat_df = pd.DataFrame(repeat_groups)
    duplicate_repeat_df = repeat_df[repeat_df['runs'] > 1].copy()
    inconsistent_repeat_df = duplicate_repeat_df[~duplicate_repeat_df['consistent']].copy()
    consistency_summary = pd.Series({
        'raw_log_rows': len(pred_log_df),
        'unique_ticker_date_forecasts': len(repeat_df),
        'duplicate_forecast_groups': len(duplicate_repeat_df),
        'duplicate_log_rows': int(duplicate_repeat_df['runs'].sum()),
        'consistent_duplicate_groups': int(duplicate_repeat_df['consistent'].sum()),
        'inconsistent_duplicate_groups': int((~duplicate_repeat_df['consistent']).sum()),
        'group_consistency_pct': duplicate_repeat_df['consistent'].mean() * 100,
        'inconsistent_groups_with_news_count_change_pct': (
            inconsistent_repeat_df['news_count_changed'].mean() * 100
            if len(inconsistent_repeat_df) else np.nan
        ),
    })
    display(consistency_summary.to_frame('value').round(2))

    if not inconsistent_repeat_df.empty:
        print('Repeated forecasts whose label changed:')
        display(inconsistent_repeat_df.sort_values(['date', 'ticker']))
    else:
        print('All repeated ticker/date forecasts kept the same label.')

## 2. Train versus validation label distribution

A model that predicts the most common class can look competent when the labels are imbalanced. This section shows the class spread overall and by ticker, then compares validation accuracy against that real validation majority baseline.

In [ ]:
def distribution_table(frame, label_column='label'):
    counts = frame[label_column].value_counts().reindex(MOVE_BINS, fill_value=0)
    result = pd.DataFrame({'count': counts, 'share_pct': (counts / max(len(frame), 1) * 100).round(2)})
    return result

train_dist = distribution_table(train_df)
val_dist = distribution_table(val_df)
distribution_comparison = train_dist[['count', 'share_pct']].add_prefix('train_').join(
    val_dist[['count', 'share_pct']].add_prefix('validation_')
)
display(distribution_comparison)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
train_dist['share_pct'].plot(kind='bar', ax=axes[0], color='#4c78a8', title='Training label share')
val_dist['share_pct'].plot(kind='bar', ax=axes[1], color='#f58518', title='Validation label share')
for ax in axes:
    ax.set_xlabel('move_bin')
    ax.set_ylabel('Percent of rows')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

ticker_label_share = (
    train_val_df.dropna(subset=['label'])
    .groupby(['split', 'ticker'])['label']
    .value_counts(normalize=True)
    .mul(100).rename('share_pct').reset_index()
)
display(ticker_label_share.sort_values(['split', 'ticker', 'label']))

## 3. Train versus validation versus production predictions

The label files alone show what the model was asked to learn; they do not show whether it fit the training examples. For a complete diagnosis, generate deterministic model predictions on both `train.jsonl` and `val.jsonl`, then compare:

- **Train predictions:** fit/memorization. High train accuracy with much lower validation accuracy indicates overfitting.
- **Validation predictions:** generalization during model development. In this project, validation loss is used for checkpoint selection and early stopping, so validation is informative but is not an untouched final test set.
- **Settled `predictions_log.jsonl`:** post-training production behavior after actual outcomes are reconciled. This is currently the strongest real-world check, although the sample is smaller and time-correlated.

The notebook currently loads `eval_predictions.jsonl` when available for validation predictions. Add a matching deterministic train evaluation output before making claims about train-versus-validation fit. A future chronological holdout or walk-forward evaluation should be reserved for the final model decision.

A prediction stream dominated by one class, especially when it is close to the relevant majority baseline, is evidence of collapse toward a maximum-signal strategy.

In [ ]:
input_status = pd.DataFrame({
    'file': ['train.jsonl', 'val.jsonl', 'predictions_log.jsonl'],
    'path': [str(TRAIN_PATH), str(VAL_PATH), str(PREDICTIONS_PATH)],
    'exists': [TRAIN_PATH.exists(), VAL_PATH.exists(), PREDICTIONS_PATH.exists()],
    'rows_loaded': [len(train_df), len(val_df), len(pred_df)],
})
display(input_status)

print('Evaluation design:')
print('- train.jsonl and val.jsonl provide the labels used during model training and checkpoint selection.')
print('- predictions_log.jsonl provides production predicted_move_bin and reconciled actual_move_bin values.')
print('- Production accuracy is calculated only from settled prediction-log rows.')
print('- No separate validation prediction file is expected or required.')

print('\nLabel distributions:')
display(distribution_comparison)

## 4. Training versus validation labels by ticker

Validation was part of model development, so this section checks whether train and validation have similar ticker coverage and label distributions. It does not claim model accuracy for validation; actual predicted-versus-actual accuracy comes from the settled production log.

In [ ]:
def ticker_split_metrics(group):
    counts = group['label'].value_counts().reindex(MOVE_BINS, fill_value=0)
    mode = counts.idxmax() if counts.sum() else 'unavailable'
    return pd.Series({
        'n': len(group),
        'mode': mode,
        'mode_share_pct': counts.max() / max(counts.sum(), 1) * 100,
        'strong_down_pct': counts['strong_down'] / max(counts.sum(), 1) * 100,
        'down_pct': counts['down'] / max(counts.sum(), 1) * 100,
        'flat_pct': counts['flat'] / max(counts.sum(), 1) * 100,
        'up_pct': counts['up'] / max(counts.sum(), 1) * 100,
        'strong_up_pct': counts['strong_up'] / max(counts.sum(), 1) * 100,
    })

train_ticker = train_df.groupby('ticker').apply(ticker_split_metrics, include_groups=False).add_prefix('train_')
val_ticker = val_df.groupby('ticker').apply(ticker_split_metrics, include_groups=False).add_prefix('validation_')
ticker_split_comparison = train_ticker.join(val_ticker, how='outer')
display(ticker_split_comparison.round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
train_ticker['train_mode_share_pct'].sort_values().plot(kind='barh', ax=axes[0], color='#4c78a8', title='Train mode share by ticker')
val_ticker['validation_mode_share_pct'].sort_values().plot(kind='barh', ax=axes[1], color='#f58518', title='Validation mode share by ticker')
for ax in axes:
    ax.set_xlabel('Mode label share (%)')
plt.tight_layout(); plt.show()

## 5. Live prediction accuracy after outcomes settle

This section uses only rows with a valid `actual_move_bin`. It reports exact five-class accuracy and directional accuracy, where down classes are negative, flat is neutral, and up classes are positive.

In [ ]:
def direction_label(value):
    if value in ['strong_down', 'down']:
        return 'down'
    if value == 'flat':
        return 'flat'
    if value in ['up', 'strong_up']:
        return 'up'
    return None

settled_df = pred_df[pred_df['settled']].copy() if not pred_df.empty else pd.DataFrame()
if not settled_df.empty:
    settled_df['correct'] = settled_df['predicted_move_bin'] == settled_df['actual_move_bin']
    settled_df['predicted_direction'] = settled_df['predicted_move_bin'].map(direction_label)
    settled_df['actual_direction'] = settled_df['actual_move_bin'].map(direction_label)
    settled_df['direction_correct'] = settled_df['predicted_direction'] == settled_df['actual_direction']
    live_mode_accuracy, live_mode = majority_accuracy(settled_df['actual_move_bin'])
    live_summary = pd.Series({
        'settled_rows': len(settled_df),
        'exact_five_class_accuracy_pct': settled_df['correct'].mean() * 100,
        'direction_accuracy_pct': settled_df['direction_correct'].mean() * 100,
        'always_actual_mode_baseline_pct': live_mode_accuracy * 100,
        'lift_over_actual_mode_points': (settled_df['correct'].mean() - live_mode_accuracy) * 100,
        'actual_mode': live_mode,
        'prediction_mode': settled_df['predicted_move_bin'].mode().iloc[0],
        'prediction_mode_share_pct': settled_df['predicted_move_bin'].value_counts(normalize=True).iloc[0] * 100,
        'prediction_entropy_bits': entropy_from_counts(settled_df['predicted_move_bin'].value_counts().reindex(MOVE_BINS, fill_value=0)),
    })
    display(live_summary.to_frame('value'))
    live_confusion = pd.crosstab(settled_df['actual_move_bin'], settled_df['predicted_move_bin']).reindex(index=MOVE_BINS, columns=MOVE_BINS, fill_value=0)
    display(live_confusion.style.background_gradient(cmap='Blues'))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(live_confusion, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_title('Live confusion matrix: actual x predicted')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    pd.DataFrame({
        'actual': settled_df['actual_move_bin'].value_counts().reindex(MOVE_BINS, fill_value=0),
        'predicted': settled_df['predicted_move_bin'].value_counts().reindex(MOVE_BINS, fill_value=0),
    }).plot(kind='bar', ax=axes[1], color=['#4c78a8', '#e45756'], title='Live actual versus predicted counts')
    axes[1].tick_params(axis='x', rotation=35)
    plt.tight_layout(); plt.show()
else:
    print('No settled live predictions are available yet.')

## 6. Slice and dice live quality by ticker

The spread columns show how outcomes differ by ticker: mean and median return, return volatility, worst/best move, exact accuracy, directional accuracy, and prediction concentration.

In [ ]:
if not settled_df.empty:
    def ticker_live_metrics(group):
        actual_counts = group['actual_move_bin'].value_counts()
        prediction_counts = group['predicted_move_bin'].value_counts()
        actual_mode_share = actual_counts.iloc[0] / len(group)
        prediction_mode_share = prediction_counts.iloc[0] / len(group)
        return pd.Series({
            'n': len(group),
            'exact_accuracy_pct': group['correct'].mean() * 100,
            'direction_accuracy_pct': group['direction_correct'].mean() * 100,
            'actual_mode': actual_counts.index[0],
            'actual_mode_share_pct': actual_mode_share * 100,
            'prediction_mode': prediction_counts.index[0],
            'prediction_mode_share_pct': prediction_mode_share * 100,
            'mean_actual_change_pct': group['actual_change_1d'].mean(),
            'median_actual_change_pct': group['actual_change_1d'].median(),
            'std_actual_change_pct': group['actual_change_1d'].std(),
            'min_actual_change_pct': group['actual_change_1d'].min(),
            'max_actual_change_pct': group['actual_change_1d'].max(),
            'mean_news_items': group['num_news_items'].mean(),
        })

    ticker_live = settled_df.groupby('ticker').apply(ticker_live_metrics, include_groups=False).sort_values('exact_accuracy_pct', ascending=False)
    display(ticker_live.round(3))
    
    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    ticker_live['exact_accuracy_pct'].sort_values().plot(kind='barh', ax=axes[0], color='#4c78a8', title='Exact accuracy by ticker')
    ticker_live['direction_accuracy_pct'].sort_values().plot(kind='barh', ax=axes[1], color='#59a14f', title='Direction accuracy by ticker')
    ticker_live['prediction_mode_share_pct'].sort_values().plot(kind='barh', ax=axes[2], color='#f28e2b', title='Prediction mode share by ticker')
    for ax in axes:
        ax.set_xlabel('Percent')
    plt.tight_layout(); plt.show()
else:
    print('No settled live predictions are available yet.')

## 7. How far off are the predictions?

There are two useful notions of error:

- **Class distance:** `strong_down` versus `down` is one step off; `strong_down` versus `strong_up` is four steps off.
- **Economic miss:** compare the actual one-day percentage return with the typical return associated with the predicted class in the training data. This is descriptive, not a claim that the class has a fixed return.

The class-distance view is available for every settled row. The return view uses the training distribution as a reference only.

In [ ]:
if not settled_df.empty:
    bin_position = {label: index for index, label in enumerate(MOVE_BINS)}
    settled_df['class_error_steps'] = (
        settled_df['predicted_move_bin'].map(bin_position) - settled_df['actual_move_bin'].map(bin_position)
    ).abs()
    settled_df['signed_class_error_steps'] = (
        settled_df['predicted_move_bin'].map(bin_position) - settled_df['actual_move_bin'].map(bin_position)
    )
    train_change_reference = train_val_df.copy()
    train_change_reference['change_1d'] = np.nan
    # The chat JSONL contains labels but generally not raw returns. Use live actual returns grouped by actual class as the empirical reference when available.
    class_return_reference = settled_df.groupby('actual_move_bin')['actual_change_1d'].agg(['count', 'mean', 'median', 'std']).reindex(MOVE_BINS)
    settled_df['predicted_class_reference_return'] = settled_df['predicted_move_bin'].map(class_return_reference['median'])
    settled_df['return_miss_vs_predicted_class_median'] = settled_df['actual_change_1d'] - settled_df['predicted_class_reference_return']
    error_summary = pd.Series({
        'mean_class_error_steps': settled_df['class_error_steps'].mean(),
        'median_class_error_steps': settled_df['class_error_steps'].median(),
        'within_one_class_step_pct': (settled_df['class_error_steps'] <= 1).mean() * 100,
        'opposite_direction_pct': ((settled_df['predicted_direction'] != settled_df['actual_direction']) & settled_df['predicted_direction'].notna() & settled_df['actual_direction'].notna()).mean() * 100,
        'mean_absolute_return_miss_vs_predicted_class_median': settled_df['return_miss_vs_predicted_class_median'].abs().mean(),
    })
    display(error_summary.round(3).to_frame('value'))
    display(class_return_reference.round(3))
    error_by_pair = settled_df.groupby(['actual_move_bin', 'predicted_move_bin']).agg(
        n=('ticker', 'size'),
        mean_actual_change_pct=('actual_change_1d', 'mean'),
        median_actual_change_pct=('actual_change_1d', 'median'),
        mean_class_error_steps=('class_error_steps', 'mean'),
    ).sort_values('n', ascending=False)
    display(error_by_pair.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.histplot(settled_df['class_error_steps'], discrete=True, ax=axes[0], color='#e45756')
    axes[0].set_title('Distance from actual class')
    axes[0].set_xlabel('Class steps off')
    sns.boxplot(data=settled_df, x='actual_move_bin', y='actual_change_1d', order=MOVE_BINS, ax=axes[1], color='#72b7b2')
    axes[1].set_title('Actual return spread by actual class')
    axes[1].tick_params(axis='x', rotation=35)
    axes[1].set_xlabel('Actual move bin'); axes[1].set_ylabel('Actual 1-day change (%)')
    plt.tight_layout(); plt.show()
else:
    print('No settled live predictions are available yet.')

## 8. Prediction behavior over time and by news volume

This helps distinguish a genuinely changing model from a fixed prior. Look for long stretches where the predicted class does not change, and check whether performance differs when the model sees zero, few, or many news items.

In [ ]:
if not settled_df.empty:
    daily_live = settled_df.groupby('date').agg(
        n=('ticker', 'size'),
        exact_accuracy_pct=('correct', lambda s: s.mean() * 100),
        direction_accuracy_pct=('direction_correct', lambda s: s.mean() * 100),
        prediction_entropy_bits=('predicted_move_bin', lambda s: entropy_from_counts(s.value_counts().reindex(MOVE_BINS, fill_value=0))),
    ).sort_index()
    display(daily_live.round(3))

    news_bins = pd.cut(settled_df['num_news_items'], bins=[-1, 0, 1, 3, 5, np.inf], labels=['0', '1', '2-3', '4-5', '6+'])
    by_news = settled_df.assign(news_bucket=news_bins).groupby('news_bucket', observed=False).agg(
        n=('ticker', 'size'),
        exact_accuracy_pct=('correct', lambda s: s.mean() * 100),
        direction_accuracy_pct=('direction_correct', lambda s: s.mean() * 100),
        mean_actual_change_pct=('actual_change_1d', 'mean'),
    )
    display(by_news.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    daily_live['exact_accuracy_pct'].plot(ax=axes[0], marker='o', title='Daily exact accuracy')
    daily_live['direction_accuracy_pct'].plot(ax=axes[0], marker='o', label='Direction accuracy')
    axes[0].set_ylabel('Percent'); axes[0].legend()
    by_news[['exact_accuracy_pct', 'direction_accuracy_pct']].plot(kind='bar', ax=axes[1], title='Accuracy by number of news items')
    axes[1].tick_params(axis='x', rotation=0)
    plt.tight_layout(); plt.show()
else:
    print('No settled live predictions are available yet.')

## 9. Review queue: largest misses

Use this table to inspect concrete failures. The most useful rows are large absolute returns with the wrong predicted direction, and class predictions that are several steps away from the actual class.

In [ ]:
if not settled_df.empty:
    review_columns = [
        'logged_at', 'ticker', 'date', 'predicted_move_bin', 'actual_move_bin',
        'actual_change_1d', 'class_error_steps', 'num_news_items', 'base_price',
    ]
    review_df = settled_df[review_columns].copy()
    review_df['abs_actual_change_pct'] = review_df['actual_change_1d'].abs()
    print('Largest absolute-return misses:')
    display(review_df.sort_values(['abs_actual_change_pct', 'class_error_steps'], ascending=False).head(25))
    print('Largest class-distance misses:')
    display(review_df.sort_values(['class_error_steps', 'abs_actual_change_pct'], ascending=False).head(25))
else:
    print('No settled live predictions are available yet.')

## Interpretation checklist

- If train accuracy is high but validation accuracy is much lower, the model is overfitting.
- If validation accuracy is near its majority baseline and prediction mode share is high, the model may be using the dominant class rather than learning useful distinctions.
- Because validation guided early stopping and best-checkpoint selection, use a future chronological holdout or walk-forward evaluation for the final go/no-go decision.
- If exact accuracy is weak but direction accuracy is materially higher, the model may understand direction but not magnitude.
- If live accuracy varies sharply by ticker, inspect ticker-specific label balance, news coverage, and volatility before changing the model.
- If the live log has few settled rows, treat all conclusions as provisional.
- The training and validation files show labels, not model predictions. Generate deterministic predictions on both splits to measure actual train fit versus validation generalization.

## 10. LLM handoff summary

Run the next cell after refreshing the notebook. It creates a compact, copyable brief for an LLM. The brief separates training/validation evidence from settled production predictions and asks for concrete recommendations about data quality, label design, model behavior, and the next training experiments.

In [ ]:
def fmt(value, digits=2):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return 'unavailable'
    if isinstance(value, (float, np.floating)):
        return f'{value:.{digits}f}'
    return str(value)

def frame_date_range(frame, column='date'):
    if frame.empty or column not in frame or frame[column].dropna().empty:
        return 'unavailable'
    values = pd.to_datetime(frame[column], errors='coerce').dropna()
    return f'{values.min().date()} to {values.max().date()}' if not values.empty else 'unavailable'

def frame_coverage(frame):
    if frame.empty:
        return {'rows': 0, 'unique_tickers': 0, 'unique_dates': 0, 'date_range': 'unavailable'}
    return {
        'rows': len(frame),
        'unique_tickers': int(frame['ticker'].nunique(dropna=True)) if 'ticker' in frame else 0,
        'unique_dates': int(frame['date'].nunique(dropna=True)) if 'date' in frame else 0,
        'date_range': frame_date_range(frame),
    }

train_mode = train_df['label'].mode().iloc[0] if not train_df.empty else 'unavailable'
val_mode = val_df['label'].mode().iloc[0] if not val_df.empty else 'unavailable'
train_mode_share = train_df['label'].value_counts(normalize=True).iloc[0] * 100 if not train_df.empty else np.nan
val_mode_share = val_df['label'].value_counts(normalize=True).iloc[0] * 100 if not val_df.empty else np.nan
train_coverage = frame_coverage(train_df)
val_coverage = frame_coverage(val_df)
production_coverage = frame_coverage(pred_df)
settled_coverage = frame_coverage(settled_df)

lines = []
lines.append('# FinCast Prediction Quality: Evidence Handoff')
lines.append('')
lines.append('Use this brief to review the model and recommend the next experiments. Separate every conclusion into: observed evidence, reasonable inference, and unknowns requiring another analysis. Do not claim that the model is overfit, leaking data, or learning useful distinctions unless the evidence supports that claim.')
lines.append('')
lines.append('## Objective')
lines.append('Assess class balance, model prediction behavior, production accuracy after outcomes settle, data-quality risks, and the smallest rigorous next experiments for this financial movement classifier.')
lines.append('')
lines.append('## Evaluation boundaries')
lines.append('- Training and validation JSONL files contain labels, not model predictions.')
lines.append('- Validation was used for loss monitoring, early stopping, and checkpoint selection; it is not an untouched final test set.')
lines.append('- Production accuracy uses the latest logged row per ticker/date, so repeated runs do not overweight one forecast.')
lines.append('- A production row is scored only when actual_move_bin is one of the five valid classes.')
lines.append('- Production rows are chronological and may be correlated; uncertainty should account for time dependence.')
lines.append('')
lines.append('## Dataset coverage')
lines.append(f'- Training coverage: {train_coverage}.')
lines.append(f'- Validation coverage: {val_coverage}.')
lines.append(f'- Production coverage after latest-run deduplication: {production_coverage}.')
lines.append(f'- Settled production coverage: {settled_coverage}.')
lines.append(f'- Source files: train={TRAIN_PATH} (exists={TRAIN_PATH.exists()}), validation={VAL_PATH} (exists={VAL_PATH.exists()}), production={PREDICTIONS_PATH} (exists={PREDICTIONS_PATH.exists()}).')
lines.append(f'- Valid movement classes, ordered by severity: {MOVE_BINS}.')
lines.append('')
lines.append('## Label balance')
lines.append(f'- Training mode: {train_mode} ({fmt(train_mode_share)}% of training labels).')
lines.append(f'- Validation mode: {val_mode} ({fmt(val_mode_share)}% of validation labels).')
lines.append(f'- Training distribution: {train_dist["count"].to_dict()}.')
lines.append(f'- Validation distribution: {val_dist["count"].to_dict()}.')
lines.append(f'- Training/validation ticker overlap: {len(set(train_df["ticker"].dropna()) & set(val_df["ticker"].dropna()))} shared tickers.')
lines.append('')
lines.append('## Repeat-run consistency')
if 'consistency_summary' in globals():
    lines.append(f'- Repeat-run summary: {consistency_summary.to_dict()}.')
    if 'inconsistent_repeat_df' in globals() and not inconsistent_repeat_df.empty:
        lines.append(f'- Inconsistent repeated forecast groups: {len(inconsistent_repeat_df)}; inspect whether news count, input text, market context, or inference settings changed.')
    else:
        lines.append('- No repeated forecast groups changed labels.')
else:
    lines.append('- Repeat-run analysis is unavailable.')
lines.append('')
lines.append('## Settled production evidence')
if not settled_df.empty:
    live_mode_accuracy, live_mode = majority_accuracy(settled_df['actual_move_bin'])
    lines.append(f'- Exact five-class accuracy: {fmt(settled_df["correct"].mean() * 100)}%.')
    lines.append(f'- Direction accuracy: {fmt(settled_df["direction_correct"].mean() * 100)}%.')
    lines.append(f'- Actual-majority baseline: {fmt(live_mode_accuracy * 100)}% ({live_mode}).')
    lines.append(f'- Lift over actual-majority baseline: {fmt((settled_df["correct"].mean() - live_mode_accuracy) * 100)} percentage points.')
    lines.append(f'- Prediction distribution: {settled_df["predicted_move_bin"].value_counts().reindex(MOVE_BINS, fill_value=0).to_dict()}.')
    lines.append(f'- Actual distribution: {settled_df["actual_move_bin"].value_counts().reindex(MOVE_BINS, fill_value=0).to_dict()}.')
    lines.append(f'- Prediction entropy: {fmt(entropy_from_counts(settled_df["predicted_move_bin"].value_counts().reindex(MOVE_BINS, fill_value=0)), 3)} bits.')
    lines.append(f'- Mean class-distance error: {fmt(settled_df["class_error_steps"].mean())}; within one class step: {fmt((settled_df["class_error_steps"] <= 1).mean() * 100)}%.')
    lines.append(f'- Opposite-direction rate: {fmt(((settled_df["predicted_direction"] != settled_df["actual_direction"]) & settled_df["predicted_direction"].notna() & settled_df["actual_direction"].notna()).mean() * 100)}%.')
    lines.append(f'- Confusion matrix, actual rows and predicted columns: {live_confusion.to_dict() if "live_confusion" in globals() else "unavailable"}.')
else:
    lines.append('- No settled production rows are available.')
lines.append('')
lines.append('## Production slices')
if not settled_df.empty:
    lines.append('- Per-ticker metrics:')
    for ticker, row in ticker_live.sort_values('exact_accuracy_pct', ascending=False).iterrows():
        lines.append(f'  - {ticker}: n={int(row["n"])}, exact={fmt(row["exact_accuracy_pct"])}%, direction={fmt(row["direction_accuracy_pct"])}%, prediction_mode={row["prediction_mode"]} ({fmt(row["prediction_mode_share_pct"])}%), actual_return_mean={fmt(row["mean_actual_change_pct"])}%, actual_return_std={fmt(row["std_actual_change_pct"])}%, mean_news_items={fmt(row["mean_news_items"])}.')
    if 'by_news' in globals():
        lines.append(f'- Accuracy by news-volume bucket: {by_news.round(3).to_dict(orient="index")}.')
    if 'daily_live' in globals():
        lines.append(f'- Daily production metrics: {daily_live.round(3).to_dict(orient="index")}.')
else:
    lines.append('- Per-ticker, daily, and news-volume slices are unavailable.')
lines.append('')
lines.append('## Error and return evidence')
if not settled_df.empty:
    lines.append(f'- Actual-return summary by actual class: {class_return_reference.round(3).to_dict(orient="index") if "class_return_reference" in globals() else "unavailable"}.')
    lines.append(f'- Most common actual/predicted error pairs: {error_by_pair.head(10).round(3).to_dict(orient="index") if "error_by_pair" in globals() else "unavailable"}.')
    lines.append('- The return comparison is descriptive only; it does not establish a tradable strategy or fixed return for a class.')
else:
    lines.append('- Error-distance and return evidence is unavailable.')
lines.append('')
lines.append('## Known constraints and pipeline context')
lines.append('- Model: phi-3-mini-4k-instruct fine-tuned with QLoRA 4-bit on a 6 GB VRAM constraint.')
lines.append('- Data: scraped daily news, sentiment, recent five-day performance, price, and volume at market close.')
lines.append('- Each training example uses only 1 to 5 news items, so news coverage and relevance may be limiting factors.')
lines.append('- Correct spelling: scraped, not scrapped.')
lines.append('')
lines.append('## Evidence currently missing')
lines.append('- Deterministic model predictions and metrics on both train.jsonl and val.jsonl.')
lines.append('- Chronological holdout or walk-forward test results.')
lines.append('- Global, per-ticker-majority, always-flat, and direction-only baselines.')
lines.append('- Bootstrap or time-aware confidence intervals for production metrics.')
lines.append('- Checkpoint name, training steps/epochs, best validation loss, seed, decoding settings, and whether inference was deterministic.')
lines.append('- Article duplicate rate, missing-value rates, news relevance quality, date/market-close alignment, weekend/holiday handling, and train/validation date overlap.')
lines.append('- Exact target construction, bin thresholds, forecast horizon, and whether any future information can enter the news or feature window.')
lines.append('')
lines.append('## Questions for the reviewing LLM')
lines.append('1. What is directly observed, what is only inferred, and what remains untestable from this brief?')
lines.append('2. Do the distributions and confusion patterns indicate class collapse, class imbalance, magnitude confusion, ticker-specific failure, or insufficient evidence?')
lines.append('3. Is five-class movement prediction appropriate for this sample size and volatility regime, compared with directional or regression objectives?')
lines.append('4. What data-quality and leakage checks should be run first, with exact pass/fail criteria?')
lines.append('5. How should the model be compared against global-majority, ticker-majority, always-flat, directional, and simple statistical baselines?')
lines.append('6. What minimum sample size and chronological coverage are needed before trusting an improvement?')
lines.append('7. Recommend the next three experiments. For each, specify the hypothesis, exact dataset/model change, metric, baseline, confidence requirement, and pass/fail criterion.')
lines.append('8. Identify any metric here that could be misleading because of deduplication, time correlation, ticker imbalance, or small samples.')
lines.append('')
llm_handoff = '\n'.join(lines)
print(llm_handoff)
HANDOFF_PATH = BASE_DIR / 'analysis' / 'prediction_quality_llm_handoff.txt'
HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
HANDOFF_PATH.write_text(llm_handoff, encoding='utf-8')
print(f'\nSaved copyable brief to: {HANDOFF_PATH}')